# 40 -- Every user who rated Rumble in the Bronx 4.0, with mu_u

Reuses notebook 37's exact data loading (train_hyperbolic.csv, the 1000x1000
chronological subset, training ratings only). For every user who rated
Rumble in the Bronx exactly 4.0 in the training set, computes n (training
rating count), the sum of training ratings, and mu_u = sum / n. Does not
touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "rumble4_user_mu_list" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_rumble4_user_mu_list.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  39  (39_step2_table_merge_breakdown.ipynb)
this notebook is:           40  (40_rumble4_user_mu_list.ipynb)


## Part 1 -- Reuse notebook 37's exact data loading

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
results37_path = out_dir / "37_step1to6_rating_habit_b_results.json"
for p in (train_path, test_path, results31_path, results37_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474, "not the section 03 split"

results31 = json.loads(results31_path.read_text())
results37 = json.loads(results37_path.read_text())
MEDIAN_USER = results31["worked_example_plus"]["user"]
assert MEDIAN_USER == 24127

movies = pd.read_csv(data_dir / "movie.csv")
rumble_row = movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)]
assert len(rumble_row) == 1
RUMBLE_ID = int(rumble_row.iloc[0]["movieId"])
RUMBLE_TITLE = rumble_row.iloc[0]["title"]
print(f"movie: {RUMBLE_TITLE} (movieId={RUMBLE_ID})")

movie: Rumble in the Bronx (Hont faan kui) (1995) (movieId=112)


## Part 2 -- Every user who rated Rumble in the Bronx 4.0 in training, with n, sum, mu_u

In [3]:
rumble4_users = train_hyp[(train_hyp["movieId"] == RUMBLE_ID) & (train_hyp["rating"] == 4.0)]["userId"].unique()
print(f"users who rated Rumble in the Bronx 4.0 in training: {len(rumble4_users)}")

records = []
for uid in rumble4_users:
    user_rows = train_hyp[train_hyp["userId"] == uid]
    n = len(user_rows)
    s = user_rows["rating"].sum()
    mu_u = s / n
    records.append({"userId": int(uid), "n": int(n), "sum": float(s), "mu_u": float(mu_u)})

records.sort(key=lambda r: r["mu_u"])
for r in records:
    print(f"user={r['userId']:>6d}  n={r['n']:>4d}  sum={r['sum']:>8.1f}  mu_u={r['mu_u']:.6f}")

assert any(r["userId"] == MEDIAN_USER for r in records), "user 24127 must be in this list"
this_user_record = next(r for r in records if r["userId"] == MEDIAN_USER)
assert abs(this_user_record["mu_u"] - results37["step1"]["mu_u"]) < 1e-6
assert this_user_record["n"] == results37["step1"]["n"]
print("\nuser 24127's n, sum, mu_u match notebook 37. OK.")

users who rated Rumble in the Bronx 4.0 in training: 97
user=  6976  n= 558  sum=  1590.5  mu_u=2.850358
user= 43545  n= 608  sum=  1736.5  mu_u=2.856086
user= 59702  n= 566  sum=  1693.0  mu_u=2.991166
user=  9545  n= 560  sum=  1724.0  mu_u=3.078571
user= 66810  n= 575  sum=  1808.5  mu_u=3.145217
user= 83949  n= 536  sum=  1715.5  mu_u=3.200560
user= 41929  n= 564  sum=  1809.0  mu_u=3.207447
user= 12793  n= 560  sum=  1798.5  mu_u=3.211607
user=106441  n= 542  sum=  1754.0  mu_u=3.236162
user=108195  n= 553  sum=  1810.5  mu_u=3.273960
user=113668  n= 625  sum=  2047.0  mu_u=3.275200
user=113181  n= 572  sum=  1894.0  mu_u=3.311189
user= 39937  n= 484  sum=  1612.0  mu_u=3.330579
user= 43657  n= 488  sum=  1637.0  mu_u=3.354508
user=103872  n= 515  sum=  1732.5  mu_u=3.364078
user=  3318  n= 636  sum=  2140.5  mu_u=3.365566
user= 81267  n= 502  sum=  1694.0  mu_u=3.374502
user= 95726  n= 478  sum=  1625.5  mu_u=3.400628
user= 25345  n= 612  sum=  2095.0  mu_u=3.423203
user= 86768  

## Part 3 -- Comparison users: lowest and highest mu_u, excluding user 24127

In [4]:
others = [r for r in records if r["userId"] != MEDIAN_USER]
print(f"other users (excluding {MEDIAN_USER}): {len(others)}")

comparison_possible = len(others) >= 2
low_user = high_user = None
if comparison_possible:
    low_user = min(others, key=lambda r: r["mu_u"])
    high_user = max(others, key=lambda r: r["mu_u"])
    print(f"lowest mu_u:  user={low_user['userId']}  mu_u={low_user['mu_u']:.6f}")
    print(f"highest mu_u: user={high_user['userId']}  mu_u={high_user['mu_u']:.6f}")
else:
    print("fewer than two other users exist; comparison is not possible.")

other users (excluding 24127): 96
lowest mu_u:  user=6976  mu_u=2.850358
highest mu_u: user=54160  mu_u=4.463415


## Part 4 -- Save the full list and the comparison result

In [5]:
results = {
    "notebook": this_name,
    "reused_from": "37_step1to6_rating_habit_b.ipynb",
    "movie": RUMBLE_TITLE,
    "movieId": RUMBLE_ID,
    "this_user": MEDIAN_USER,
    "n_users_rated_4": len(records),
    "users_sorted_by_mu_u": records,
    "comparison_possible": comparison_possible,
    "low_mu_u_user": low_user,
    "high_mu_u_user": high_user,
}
results_path = out_dir / "40_rumble4_user_mu_list_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps({k: v for k, v in results.items() if k != "users_sorted_by_mu_u"}, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/40_rumble4_user_mu_list_results.json
{
  "notebook": "40_rumble4_user_mu_list.ipynb",
  "reused_from": "37_step1to6_rating_habit_b.ipynb",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "movieId": 112,
  "this_user": 24127,
  "n_users_rated_4": 97,
  "comparison_possible": true,
  "low_mu_u_user": {
    "userId": 6976,
    "n": 558,
    "sum": 1590.5,
    "mu_u": 2.850358422939068
  },
  "high_mu_u_user": {
    "userId": 54160,
    "n": 492,
    "sum": 2196.0,
    "mu_u": 4.463414634146342
  }
}
